In [1]:
# objective is to find the genomic coordinates of all the genes in ranked_genes_list.txt

In [15]:
library(org.Hs.eg.db)
library(GenomicFeatures)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(clusterProfiler)
library(rtracklayer)

In [3]:
txdb <- TxDb.Hsapiens.UCSC.hg38.knownGene
#genes_gr <- genes(txdb)

  2162 genes were dropped because they have exons located on both strands
  of the same reference sequence or on more than one reference sequence,
  so cannot be represented by a single genomic range.
  Use 'single.strand.genes.only=FALSE' to get all the genes in a
  GRangesList object, or use suppressMessages() to suppress this message.



In [4]:
genes_gr <- genes(txdb, single.strand.genes.only = FALSE)
genes_gr

GRangesList object of length 33131:
$`1`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr19 58345178-58362751      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`10`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]     chr8 18386311-18401218      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`100`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr20 44584896-44652252      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

...
<33128 more elements>

In [5]:
gene_df <- read.table("/usr/users/papantonis1/aman/rnaseq_data/ranked_gene_list.txt", header=TRUE)

gene_df$ensembl_clean <- sub("\\..*", "", gene_df$ensembl)

# Map to entrezid
mapped <- bitr(gene_df$ensembl_clean,
               fromType = "ENSEMBL",
               toType = "ENTREZID",
               OrgDb = org.Hs.eg.db)
mapped

'select()' returned 1:many mapping between keys and columns

Warning message in bitr(gene_df$ensembl_clean, fromType = "ENSEMBL", toType = "ENTREZID", :
“46.97% of input gene IDs are fail to map...”


,ENSEMBL,ENTREZID
,<chr>,<chr>
1,ENSG00000290825,84771
2,ENSG00000290825,727856
3,ENSG00000290825,100287102
4,ENSG00000290825,100287596
5,ENSG00000290825,102725121
12,ENSG00000222623,106480049
13,ENSG00000222623,124906683
15,ENSG00000292994,127239154
18,ENSG00000293331,101928626


In [6]:
#mapped$ENTREZID

In [7]:
my_de <- mapped$ENTREZID
genes_gr <- genes_gr[names(genes_gr) %in% my_de]
genes_gr

GRangesList object of length 22612:
$`1`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr19 58345178-58362751      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`10`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]     chr8 18386311-18401218      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`100`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr20 44584896-44652252      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

...
<22609 more elements>

In [14]:
head(unlist(genes_gr)) # unlist to flat

GRanges object with 6 ranges and 0 metadata columns:
                   seqnames              ranges strand
                      <Rle>           <IRanges>  <Rle>
      1               chr19   58345178-58362751      -
     10                chr8   18386311-18401218      +
    100               chr20   44584896-44652252      -
   1000               chr18   27932879-28177946      -
  10000                chr1 243488233-243851079      -
  10000 chr1_KI270763v1_alt       500341-867542      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

In [16]:
# export
ranked_gene_list_bed <- unlist(genes_gr)
export(ranked_gene_list_bed, con = "genes_output.bed", format = "BED")

In [17]:
rg_bed = read.table("genes_output.bed")
head(rg_bed)
# v4 - granges list ele. name, v5 - score, v6 - strand

,V1,V2,V3,V4,V5,V6
,<chr>,<int>,<int>,<int>,<int>,<chr>
1,chr19,58345177,58362751,1,0,-
2,chr8,18386310,18401218,10,0,+
3,chr20,44584895,44652252,100,0,-
4,chr18,27932878,28177946,1000,0,-
5,chr1,243488232,243851079,10000,0,-
6,chr1_KI270763v1_alt,500340,867542,10000,0,-


#### Upregulated genes - DeSeq2

In [21]:
upreg_gene_df <- read.table("/usr/users/papantonis1/aman/rnaseq_data/upregulated_genes.txt", header=TRUE)

upreg_gene_df$ensembl_clean <- sub("\\..*", "", upreg_gene_df$ensembl)

# Map to entrezid
mapped_upreg <- bitr(upreg_gene_df$ensembl_clean,
               fromType = "ENSEMBL",
               toType = "ENTREZID",
               OrgDb = org.Hs.eg.db)
mapped_upreg

'select()' returned 1:many mapping between keys and columns

Warning message in bitr(upreg_gene_df$ensembl_clean, fromType = "ENSEMBL", toType = "ENTREZID", :
“0.44% of input gene IDs are fail to map...”


,ENSEMBL,ENTREZID
,<chr>,<chr>
1,ENSG00000078808,51150
2,ENSG00000049239,9563
3,ENSG00000171603,22883
4,ENSG00000142657,5226
5,ENSG00000083444,5351
6,ENSG00000116731,7799
7,ENSG00000197312,84301
8,ENSG00000142627,1969
9,ENSG00000127463,23065


In [22]:
upreg_de_genes <- mapped_upreg$ENTREZID
genes_gr_overlap_upreg <- genes_gr[names(genes_gr) %in% upreg_de_genes]
genes_gr_overlap_upreg

GRangesList object of length 681:
$`1001`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr16 68644993-68727468      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`100128966`
GRanges object with 1 range and 0 metadata columns:
      seqnames              ranges strand
         <Rle>           <IRanges>  <Rle>
  [1]     chr5 138332105-138337146      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`100652740`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr16 31201885-31203452      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

...
<678 more elements>

In [23]:
head(unlist(genes_gr_overlap_upreg))

GRanges object with 6 ranges and 0 metadata columns:
            seqnames              ranges strand
               <Rle>           <IRanges>  <Rle>
       1001    chr16   68644993-68727468      +
  100128966     chr5 138332105-138337146      +
  100652740    chr16   31201885-31203452      +
      10076     chr1   29236516-29326813      +
  100861548     chr1   20642657-20652193      -
      10097     chr2   65227788-65271253      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

In [24]:
# export
upreg_gene_bed <- unlist(genes_gr_overlap_upreg)
export(upreg_gene_bed, con = "upreg_genes.bed", format = "BED")

In [25]:
upreg_bed = read.table("upreg_genes.bed")
head(upreg_bed)
# v4 - granges list ele. name, v5 - score, v6 - strand

,V1,V2,V3,V4,V5,V6
,<chr>,<int>,<int>,<int>,<int>,<chr>
1,chr16,68644992,68727468,1001,0,+
2,chr5,138332104,138337146,100128966,0,+
3,chr16,31201884,31203452,100652740,0,+
4,chr1,29236515,29326813,10076,0,+
5,chr1,20642656,20652193,100861548,0,-
6,chr2,65227787,65271253,10097,0,+


#### Downregulated genes

In [26]:
downreg_gene_df <- read.table("/usr/users/papantonis1/aman/rnaseq_data/downregulated_genes.txt", header=TRUE)

downreg_gene_df$ensembl_clean <- sub("\\..*", "", downreg_gene_df$ensembl)

# Map to entrezid
mapped_downreg <- bitr(downreg_gene_df$ensembl_clean,
               fromType = "ENSEMBL",
               toType = "ENTREZID",
               OrgDb = org.Hs.eg.db)
mapped_downreg

'select()' returned 1:many mapping between keys and columns

Warning message in bitr(downreg_gene_df$ensembl_clean, fromType = "ENSEMBL", toType = "ENTREZID", :
“8.43% of input gene IDs are fail to map...”


,ENSEMBL,ENTREZID
,<chr>,<chr>
1,ENSG00000187961,339451
2,ENSG00000131584,116983
3,ENSG00000107404,1855
4,ENSG00000221978,81669
5,ENSG00000197530,142678
6,ENSG00000158286,388591
7,ENSG00000142733,9064
8,ENSG00000200087,26768
9,ENSG00000197989,85028


In [27]:
downreg_de_genes <- mapped_downreg$ENTREZID
genes_gr_overlap_downreg <- genes_gr[names(genes_gr) %in% downreg_de_genes]
genes_gr_overlap_downreg

GRangesList object of length 153:
$`100302743`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]     chr2 10446714-10446849      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`100533107`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr20 63658294-63698684      +
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

$`10139`
GRanges object with 1 range and 0 metadata columns:
      seqnames            ranges strand
         <Rle>         <IRanges>  <Rle>
  [1]    chr20 63698642-63708025      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

...
<150 more elements>

In [28]:
head(unlist(genes_gr_overlap_downreg))
downreg_gene_bed <- unlist(genes_gr_overlap_downreg)
export(downreg_gene_bed, con = "downreg_genes.bed", format = "BED")

GRanges object with 6 ranges and 0 metadata columns:
            seqnames            ranges strand
               <Rle>         <IRanges>  <Rle>
  100302743     chr2 10446714-10446849      -
  100533107    chr20 63658294-63698684      +
      10139    chr20 63698642-63708025      -
      10142     chr7 91940840-92110673      +
  101927480    chr16 54845189-54874168      -
      10529    chr10 20779973-21293011      -
  -------
  seqinfo: 711 sequences (1 circular) from hg38 genome

In [29]:
downreg_bed = read.table("downreg_genes.bed")
head(downreg_bed)
# v4 - granges list ele. name, v5 - score, v6 - strand

,V1,V2,V3,V4,V5,V6
,<chr>,<int>,<int>,<int>,<int>,<chr>
1,chr2,10446713,10446849,100302743,0,-
2,chr20,63658293,63698684,100533107,0,+
3,chr20,63698641,63708025,10139,0,-
4,chr7,91940839,92110673,10142,0,+
5,chr16,54845188,54874168,101927480,0,-
6,chr10,20779972,21293011,10529,0,-


----

In [8]:
mega_loops_ctrl <- read.table("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/megalooplist/mega_merged_loops_ctrl.bedpe", header=FALSE)
mega_loops_rbp1 <- read.table("/usr/users/papantonis1/aman/microc_project/loop_calling_premade_hic/merged_loops_3tools/megalooplist/mega_merged_loops_rbp1.bedpe", header=FALSE)
mega_loops_ctrl

V1,V2,V3,V4,V5,V6
<chr>,<int>,<int>,<chr>,<int>,<int>
chr1,3912500,3917500,chr1,4592500,4597500
chr1,5502500,5507500,chr1,5622500,5627500
chr1,5502500,5507500,chr1,5742500,5747500
chr1,5632500,5637500,chr1,5742500,5747500
chr1,5635000,5640000,chr1,5980000,5985000
chr1,6555000,6560000,chr1,6645000,6650000
chr1,6720000,6725000,chr1,6885000,6890000
chr1,6802500,6807500,chr1,6892500,6897500
chr1,7032500,7037500,chr1,7202500,7207500


In [9]:
anchor1_ctrl <- GRanges(seqnames = mega_loops_ctrl$V1,
                   ranges = IRanges(start = mega_loops_ctrl$V2, end = mega_loops_ctrl$V3))

anchor2_ctrl <- GRanges(seqnames = mega_loops_ctrl$V4,
                   ranges = IRanges(start = mega_loops_ctrl$V5, end = mega_loops_ctrl$V6))

anchor2_ctrl

GRanges object with 19036 ranges and 0 metadata columns:
          seqnames            ranges strand
             <Rle>         <IRanges>  <Rle>
      [1]     chr1   4592500-4597500      *
      [2]     chr1   5622500-5627500      *
      [3]     chr1   5742500-5747500      *
      [4]     chr1   5742500-5747500      *
      [5]     chr1   5980000-5985000      *
      ...      ...               ...    ...
  [19032]     chrY 11722500-11727500      *
  [19033]     chrY 11722500-11727500      *
  [19034]     chrY 11712500-11717500      *
  [19035]     chrY 11680000-11685000      *
  [19036]     chrY 20732500-20737500      *
  -------
  seqinfo: 24 sequences from an unspecified genome; no seqlengths

In [10]:
anchor1_rbp1 <- GRanges(seqnames = mega_loops_rbp1$V1,
                   ranges = IRanges(start = mega_loops_rbp1$V2, end = mega_loops_rbp1$V3))

anchor2_rbp1 <- GRanges(seqnames = mega_loops_rbp1$V4,
                   ranges = IRanges(start = mega_loops_rbp1$V5, end = mega_loops_rbp1$V6))

anchor2_rbp1

GRanges object with 16599 ranges and 0 metadata columns:
          seqnames            ranges strand
             <Rle>         <IRanges>  <Rle>
      [1]     chr1   4065000-4070000      *
      [2]     chr1   4155000-4160000      *
      [3]     chr1   5622500-5627500      *
      [4]     chr1   5732500-5737500      *
      [5]     chr1   5752500-5757500      *
      ...      ...               ...    ...
  [16595]     chrY 11722500-11727500      *
  [16596]     chrY 11712500-11717500      *
  [16597]     chrY 11680000-11685000      *
  [16598]     chrY 11760000-11765000      *
  [16599]     chrY 20742500-20747500      *
  -------
  seqinfo: 24 sequences from an unspecified genome; no seqlengths

In [11]:
hits_a1_ctrl <- findOverlaps(anchor1_ctrl, genes_gr)
hits_a2_ctrl <- findOverlaps(anchor2_ctrl, genes_gr)
hits_a1_ctrl

Hits object with 12515 hits and 0 metadata columns:
          queryHits subjectHits
          <integer>   <integer>
      [1]         1         285
      [2]         6       19849
      [3]         8        8521
      [4]         9        8521
      [5]        10        8521
      ...       ...         ...
  [12511]     19021       15720
  [12512]     19022       11287
  [12513]     19027       14204
  [12514]     19028         351
  [12515]     19036        8897
  -------
  queryLength: 19036 / subjectLength: 22612

In [12]:
hits_a1_rbp1 <- findOverlaps(anchor1_rbp1, genes_gr)
hits_a2_rbp1 <- findOverlaps(anchor2_rbp1, genes_gr)
hits_a1_rbp1

Hits object with 10863 hits and 0 metadata columns:
          queryHits subjectHits
          <integer>   <integer>
      [1]         1         285
      [2]         2         285
      [3]         9        8521
      [4]        10        8521
      [5]        11        4575
      ...       ...         ...
  [10859]     16574        7306
  [10860]     16579       18744
  [10861]     16583        9820
  [10862]     16584        3005
  [10863]     16587         351
  -------
  queryLength: 16599 / subjectLength: 22612

In [13]:
hits_a2_ctrl

Hits object with 12655 hits and 0 metadata columns:
          queryHits subjectHits
          <integer>   <integer>
      [1]         5        9463
      [2]         6       15466
      [3]         7        8521
      [4]         8        8521
      [5]         9        8521
      ...       ...         ...
  [12651]     19023       13825
  [12652]     19023       14378
  [12653]     19025       11104
  [12654]     19027        2092
  [12655]     19027       16448
  -------
  queryLength: 19036 / subjectLength: 22612